# Combining features in Inception, ResNet, and DenseNet

How does a CNN combine features when more than one path leads to the next layer?
We will send the same small batch through three kinds of CNN blocks and print
their output shapes. We will also run a backward pass to make sure gradients
reach each block's trainable weights. This notebook uses only PyTorch and runs
on a CPU.

In [ ]:
import torch
from torch import nn

torch.manual_seed(7)
torch.set_num_threads(min(4, torch.get_num_threads()))

features = torch.randn(2, 8, 16, 16, device="cpu")
print("Input (batch, channels, height, width):", tuple(features.shape))

## Combining parallel branches in Inception

An Inception module can use several operations on the same input. Here, one
branch uses a 1 × 1 convolution, two branches look at wider neighborhoods, and
the last branch uses max pooling. A 1 × 1 convolution mixes channels at each
position. We use one before each wider convolution to give it fewer channels
to work on.

The padding keeps all four outputs at 16 × 16. If the branches produce 4, 6, 5,
and 3 channels, how many channels will we get by placing the outputs next to
one another along the channel dimension?

In [ ]:
def conv_bn_relu(in_channels, out_channels, kernel_size, padding=0):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
    )


class InceptionBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branch1 = conv_bn_relu(in_channels, 4, kernel_size=1)
        self.branch3 = nn.Sequential(
            conv_bn_relu(in_channels, 4, kernel_size=1),
            conv_bn_relu(4, 6, kernel_size=3, padding=1),
        )
        self.branch5 = nn.Sequential(
            conv_bn_relu(in_channels, 4, kernel_size=1),
            conv_bn_relu(4, 5, kernel_size=5, padding=2),
        )
        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            conv_bn_relu(in_channels, 3, kernel_size=1),
        )

    def forward(self, x):
        branches = (self.branch1(x), self.branch3(x), self.branch5(x), self.branch_pool(x))
        return torch.cat(branches, dim=1)

In [ ]:
inception = InceptionBlock(in_channels=8)
inception_output = inception(features)
assert inception_output.shape == (2, 18, 16, 16)
print("Inception output:", tuple(inception_output.shape))

We get 18 channels because every branch contributes its own channels to the
output. This is *concatenation*. A ResNet block also combines two paths, but
there the two paths contribute to the same set of output channels.

## Adding a shortcut in ResNet

A residual block sends the input through two convolutions and adds the result
to a shortcut path. When both paths already have the same shape, the shortcut
can pass through the input as it is. This is called an *identity shortcut*.

What if we want half the height and width and twice the channels? In that case
we use a 1 × 1 convolution with stride 2 on the shortcut, so the two paths
still line up when we add them. This is called a *projection shortcut*.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        return torch.relu(self.residual(x) + self.shortcut(x))

In [ ]:
identity_block = ResidualBlock(in_channels=8, out_channels=8)
projection_block = ResidualBlock(in_channels=8, out_channels=16, stride=2)
identity_output = identity_block(features)
projection_output = projection_block(features)

assert identity_output.shape == features.shape
assert projection_output.shape == (2, 16, 8, 8)
print("ResNet with identity shortcut:  ", tuple(identity_output.shape))
print("ResNet with projection shortcut:", tuple(projection_output.shape))

Addition produces one value at each matching position and channel. The
projected block has 16 output channels because each path has 16 channels after
its convolutions. DenseNet combines paths differently. It keeps the earlier
features available as separate channels for later layers.

## Keeping earlier features in DenseNet

In a dense block, each layer receives the original input and the features
produced by the preceding layers in that block. Each layer creates a fixed
number of new channels, called the *growth rate*. We will use a growth rate
of 4. Our first layer turns 8 channels into 12 by adding four new channels
beside the original eight. The next layer receives those 12 channels and
contributes another four.

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        bottleneck_channels = 4 * growth_rate
        self.new_features = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(),
            nn.Conv2d(in_channels, bottleneck_channels, 1, bias=False),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(),
            nn.Conv2d(bottleneck_channels, growth_rate, 3, padding=1, bias=False),
        )

    def forward(self, x):
        return torch.cat((x, self.new_features(x)), dim=1)


class TransitionBlock(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.AvgPool2d(kernel_size=2, stride=2),
        )

In [ ]:
dense_first = DenseLayer(in_channels=8, growth_rate=4)
dense_second = DenseLayer(in_channels=12, growth_rate=4)
transition = TransitionBlock(in_channels=16, out_channels=8)

dense_first_output = dense_first(features)
dense_second_output = dense_second(dense_first_output)
transition_output = transition(dense_second_output)

assert dense_first_output.shape == (2, 12, 16, 16)
assert dense_second_output.shape == (2, 16, 16, 16)
torch.testing.assert_close(dense_second_output[:, :8], features)
assert transition_output.shape == (2, 8, 8, 8)
print("First dense layer:  ", tuple(dense_first_output.shape))
print("Second dense layer: ", tuple(dense_second_output.shape))
print("After transition:  ", tuple(transition_output.shape))

The first eight channels still hold the original input when the second layer
finishes. Between dense blocks, a transition block can reduce this growing
channel count with a 1 × 1 convolution and shrink the grid with average pooling.

## Checking the backward pass

We can check whether gradients reach the trainable weights with a simple
stand-in for a loss. In each example below we take the mean of the squared
outputs and call `backward()`. Every block should receive finite gradients.

In [ ]:
examples = {
    "Inception": inception,
    "ResNet identity": identity_block,
    "ResNet projection": projection_block,
    "DenseNet with transition": nn.Sequential(dense_first, dense_second, transition),
}

for name, block in examples.items():
    block.zero_grad(set_to_none=True)
    block(features).square().mean().backward()
    for parameter in block.parameters():
        assert parameter.grad is not None
        assert torch.isfinite(parameter.grad).all()
    print(f"{name}: finite parameter gradients")

The same input led to three different kinds of feature combinations. Inception
concatenated the outputs of parallel branches. ResNet added the shortcut to
the residual path. DenseNet kept the input channels and appended new ones at
each layer. When you read an architecture diagram, follow the arrows into the
point where paths meet and ask which of these operations it shows.